# Pig Posture Recognition - V6 Training

**Ensemble-Ansatz: EVA-02 Large (ViT) + ConvNeXt V2 Large (CNN)** — Alternative zu V5 mit staerkerem ViT.

**Unterschied zu V5:**
- V5 = DINOv2 Large @ 518 (self-distillation pretrained)
- V6 = **EVA-02 Large @ 448** (MIM-pretrained auf 38M Bildern, oft SOTA auf ImageNet)

Rest identisch: V3-Hyperparams (LR=1e-4, CE+class_weights), EMA, MixUp/CutMix,
Heavy Augs, Layer-wise LR Decay 0.75, Strict CLO auf Test-Cams.

**Warum EVA-02 statt DINOv2?** Staerkere Features laut Benchmarks (+1-2% ImageNet).
**Warum nicht statt DINOv2 behalten?** Beide sind ViTs, also weniger Diversitaet im Ensemble
als DINOv2+ConvNeXt. V5 vs V6 zeigt welcher ViT-Stil fuer diesen Use-Case besser passt.

## Configuration

In [1]:
TAG = "T2"   # "T1" oder "T2"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

_candidates = [
    "multiview_pig_posture_recognition",
    "./multiview_pig_posture_recognition",
    "/datasets/multi-view-pig-posture-recognition",
    "/multi-view-pig-posture-recognition",
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None, "Datenverzeichnis nicht gefunden!"
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)}")

if TAG == "T1":
    CSV_PATH = f"{DATA_ROOT}/train1.csv"
    IMG_DIR  = f"{DATA_ROOT}/train1_images"
else:
    CSV_PATH = f"{DATA_ROOT}/train2.csv"
    IMG_DIR  = f"{DATA_ROOT}/train2_images"

OUTPUT_DIR = f"runs/v6_{TAG.lower()}"

# --- Architekturen: EVA-02 Large (ViT) + ConvNeXt V2 Large (CNN) ---
# 4x Tesla V100 32GB SXM2 (128 GB total VRAM)
# EVA-02 Large @ 448:      ~305M params; batch 16 global (4/GPU), grad_accum 2 -> eff. 32
# ConvNeXt V2 Large @ 384: ~200M params; batch 24 global (6/GPU), kein accum
ARCH_LIST = [
    {
        "name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
        "img_size": 448,          # native EVA-02 Aufloesung; 448/14 = 32 tokens
        "batch_size": 16,         # 4 pro GPU
        "grad_accum": 2,          # effektiv 32
        "lr": 1e-4,
        "lr_backbone_mult": 0.1,
        "layer_decay": 0.75,      # ViT-typisch
        "epochs": 30,
        "patience": 10,
        "prefix": "eva02l",
    },
    {
        "name": "convnextv2_large.fcmae_ft_in22k_in1k_384",
        "img_size": 384,
        "batch_size": 24,
        "grad_accum": 1,
        "lr": 1e-4,
        "lr_backbone_mult": 0.1,
        "layer_decay": None,
        "epochs": 30,
        "patience": 10,
        "prefix": "convnextv2l",
    },
]

# --- Training Hyperparams (global) ---
WARMUP_EPOCHS     = 3
LABEL_SMOOTH      = 0.05
PAD_RATIO         = 0.1
NUM_WORKERS       = 16
SEED              = 42
NUM_CLASSES       = 5

# MixUp + CutMix (timm.data.Mixup)
MIXUP_ALPHA       = 0.2
CUTMIX_ALPHA      = 1.0
MIX_PROB          = 0.5   # Wahrscheinlichkeit ueberhaupt zu mixen
SWITCH_PROB       = 0.5   # Wahrscheinlichkeit CutMix statt MixUp

# EMA
USE_EMA           = True
EMA_DECAY         = 0.9995

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

# --- Loss: V3-Stil, CE + class weights (KEIN Focal) ---
LOSS_TYPE    = "ce"

# --- Test-Kameras ---
TEST_CAMERAS = ["pen1_tur_cam1", "pen2_orb_cam2", "pen2_tur_cam2"]

# --- Validation: strict (nur Test-Cams, 3 Folds) ---
VALIDATION_STRATEGY = "test_only"

# --- Pseudo-Labels ---
USE_PSEUDO_LABELS = False
PSEUDO_CSV        = None

PRETRAINED_CKPT   = None

print(f"Tag: {TAG}  |  Architekturen: {len(ARCH_LIST)}")
for a in ARCH_LIST:
    print(f"  - {a['prefix']}: {a['name']} @ {a['img_size']}px  |  BS={a['batch_size']}*{a['grad_accum']}  LR={a['lr']}")
print(f"Validation: {VALIDATION_STRATEGY}  |  Loss: {LOSS_TYPE}  |  EMA: {USE_EMA}  |  Output: {OUTPUT_DIR}")

DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  Architekturen: 2
  - eva02l: eva02_large_patch14_448.mim_m38m_ft_in22k_in1k @ 448px  |  BS=16*2  LR=0.0001
  - convnextv2l: convnextv2_large.fcmae_ft_in22k_in1k_384 @ 384px  |  BS=24*1  LR=0.0001
Validation: test_only  |  Loss: ce  |  EMA: True  |  Output: runs/v6_t2


## Imports

In [2]:
import os, ast, random, re, copy
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import torchvision.transforms.functional as TFn
import timm
from timm.data import Mixup

from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} ({vram:.1f} GB)")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

<jemalloc>: Unsupported system page size


Device: cuda
  GPU 0: Tesla V100-SXM2-32GB (33.8 GB)
  GPU 1: Tesla V100-SXM2-32GB (33.8 GB)
  GPU 2: Tesla V100-SXM2-32GB (33.8 GB)
  GPU 3: Tesla V100-SXM2-32GB (33.8 GB)


## Daten laden

In [3]:
df = pd.read_csv(CSV_PATH)

def extract_camera(image_id):
    m = re.match(r"(pen\d+_\w+_cam\d+)", image_id)
    return m.group(1) if m else "unknown"

df["camera"] = df["image_id"].apply(extract_camera)
df["img_dir"] = IMG_DIR

if USE_PSEUDO_LABELS and PSEUDO_CSV and os.path.exists(PSEUDO_CSV):
    pseudo_df = pd.read_csv(PSEUDO_CSV)
    keep_cols = [c for c in pseudo_df.columns if c in ["row_id","image_id","width","height","bbox","class_id"]]
    pseudo_df = pseudo_df[keep_cols]
    pseudo_df["camera"] = pseudo_df["image_id"].apply(extract_camera)
    pseudo_df["img_dir"] = os.path.join(DATA_ROOT, "test_images")
    df = pd.concat([df, pseudo_df], ignore_index=True)
    print(f"Pseudo-Labels: {len(pseudo_df)} hinzugefuegt -> Gesamt: {len(df)}")

print(f"Instanzen: {len(df)}  |  Bilder: {df['image_id'].nunique()}")
print(f"Kameras ({df['camera'].nunique()}):")
for cam in sorted(df["camera"].unique()):
    cnt = (df["camera"] == cam).sum()
    print(f"  {cam:<25} {cnt:>5} ({100*cnt/len(df):.1f}%)")
print(f"Klassen:")
for c in range(NUM_CLASSES):
    cnt = (df["class_id"] == c).sum()
    print(f"  {c} - {CLASS_NAMES[c]:<22} {cnt:>5} ({100*cnt/len(df):.1f}%)")

Instanzen: 23450  |  Bilder: 3150
Kameras (8):
  pen1_orb_cam1               722 (3.1%)
  pen1_orb_cam2              1488 (6.3%)
  pen1_tur_cam1               200 (0.9%)
  pen1_tur_cam2              8194 (34.9%)
  pen2_orb_cam1              2817 (12.0%)
  pen2_orb_cam2               120 (0.5%)
  pen2_tur_cam1              9713 (41.4%)
  pen2_tur_cam2               196 (0.8%)
Klassen:
  0 - Lateral_lying_left      3083 (13.1%)
  1 - Lateral_lying_right     3435 (14.6%)
  2 - Sitting                  695 (3.0%)
  3 - Standing                9928 (42.3%)
  4 - Sternal_lying           6309 (26.9%)


## Dataset mit Label-aware Flip

In [4]:
class PigPostureDataset(Dataset):
    def __init__(self, df, transform=None, pad_ratio=0.25, is_train=False, hflip_prob=0.5):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.pad_ratio = pad_ratio
        self.is_train = is_train
        self.hflip_prob = hflip_prob

    def __len__(self):
        return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(row["img_dir"], row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        label = int(row["class_id"])

        if self.is_train and random.random() < self.hflip_prob:
            crop = TFn.hflip(crop)
            if label == 0: label = 1
            elif label == 1: label = 0

        if self.transform:
            crop = self.transform(crop)
        return crop, label

## Heavy Augmentations (V5)

Gezielt gegen Kamera-Domain-Shift:
- **RandomResizedCrop(0.7-1.0)**: simuliert leichte BBox-Variation
- **Stronger Perspective** (0.15, p=0.4): verschiedene Kamerawinkel
- **GaussianBlur (p=0.2)**: unterschiedliche Schaerfe
- **CameraSimTransform**: JPEG-Artefakte, Rauschen, Aufloesungs-Downsampling
- **Color/Gray/Erasing**: wie V4

In [5]:
class CameraSimTransform:
    def __init__(self, p=0.4):
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        if random.random() < 0.4:
            w, h = img.size
            scale = random.uniform(0.3, 0.7)
            small = img.resize((max(16, int(w*scale)), max(16, int(h*scale))), Image.BILINEAR)
            img = small.resize((w, h), Image.BILINEAR)
        if random.random() < 0.2:
            quality = random.randint(25, 65)
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=quality)
            buffer.seek(0)
            img = Image.open(buffer).convert("RGB")
        if random.random() < 0.15:
            arr = np.array(img, dtype=np.float32)
            noise = np.random.normal(0, random.uniform(5, 15), arr.shape)
            arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)
        return img


class PerspectiveJitter:
    def __init__(self, distortion_scale=0.15, p=0.4):
        self.distortion_scale = distortion_scale
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            d = self.distortion_scale
            w, h = img.size
            return TFn.perspective(
                img,
                startpoints=[[0,0],[w,0],[w,h],[0,h]],
                endpoints=[
                    [int(random.uniform(0, w*d)), int(random.uniform(0, h*d))],
                    [int(w - random.uniform(0, w*d)), int(random.uniform(0, h*d))],
                    [int(w - random.uniform(0, w*d)), int(h - random.uniform(0, h*d))],
                    [int(random.uniform(0, w*d)), int(h - random.uniform(0, h*d))],
                ],
                fill=0
            )
        return img


def get_train_transform(size):
    return T.Compose([
        CameraSimTransform(p=0.4),
        PerspectiveJitter(distortion_scale=0.15, p=0.4),
        T.Resize((int(size*1.15), int(size*1.15)), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomResizedCrop(size, scale=(0.7, 1.0), ratio=(0.85, 1.15),
                            interpolation=T.InterpolationMode.BICUBIC),
        T.RandomRotation(degrees=12),
        T.RandomAffine(degrees=0, scale=(0.85, 1.15), translate=(0.05, 0.05)),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.08),
        T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.2),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])


def get_val_transform(size):
    return T.Compose([
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

print("Heavy Augmentations definiert.")

Heavy Augmentations definiert.


## EMA (Exponential Moving Average) Weights

Fuehrt parallel eine geglaettete Version der Model-Gewichte - reduziert Noise
am Ende des Trainings. Sehr billig (1 Copy) und oft +0.5-1% F1.

In [6]:
class ModelEMA:
    """EMA fuer Model-Weights. State wird auf CPU gehalten wenn cpu=True."""
    def __init__(self, model, decay=0.9995, cpu=False):
        self.decay = decay
        self.cpu = cpu
        inner = model.module if isinstance(model, nn.DataParallel) else model
        self.ema = copy.deepcopy(inner).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        if cpu:
            self.ema = self.ema.cpu()

    @torch.no_grad()
    def update(self, model):
        inner = model.module if isinstance(model, nn.DataParallel) else model
        msd = inner.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                src = msd[k].detach()
                if self.cpu:
                    src = src.cpu()
                v.mul_(self.decay).add_(src, alpha=1 - self.decay)
            else:
                v.copy_(msd[k])

    def state_dict(self):
        return self.ema.state_dict()

## Layer-wise LR Decay (ViT)

Fruehe Layers bekommen kleineres LR als spaetere.
`lr_layer_i = base_lr * decay ^ (num_layers - i)` - ViT-typisch decay=0.75.
Fuer ConvNeXt wird fallback auf einfaches backbone_mult genutzt.

In [7]:
def get_vit_layer_id(name, num_layers):
    """Zaehlt Layers fuer ViT (DINOv2). Returns layer_id von 0 (patch_embed) bis num_layers."""
    name = name.replace("module.", "")
    if name.startswith(("cls_token", "pos_embed", "patch_embed", "mask_token", "reg_token")):
        return 0
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1
    return num_layers  # head, norm, etc.


def build_vit_param_groups(model, base_lr, layer_decay=0.75, weight_decay=1e-2):
    """Layer-wise LR decay fuer ViT. head/norm bekommt volles LR, patch_embed das niedrigste."""
    inner = model.module if isinstance(model, nn.DataParallel) else model
    num_layers = len(inner.blocks) if hasattr(inner, "blocks") else 12
    lr_scales = [layer_decay ** (num_layers - i) for i in range(num_layers + 1)]

    groups = {}
    for name, param in inner.named_parameters():
        if not param.requires_grad:
            continue
        lid = get_vit_layer_id(name, num_layers)
        no_wd = param.ndim <= 1 or name.endswith(".bias") or "norm" in name or "cls_token" in name or "pos_embed" in name
        key = (lid, no_wd)
        if key not in groups:
            groups[key] = {
                "params": [],
                "lr": base_lr * lr_scales[lid],
                "weight_decay": 0.0 if no_wd else weight_decay,
                "lr_scale": lr_scales[lid],
            }
        groups[key]["params"].append(param)
    return list(groups.values()), num_layers


def build_convnext_param_groups(model, base_lr, backbone_mult=0.1, weight_decay=1e-2):
    inner = model.module if isinstance(model, nn.DataParallel) else model
    head_params, backbone_params = [], []
    for name, param in inner.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("head") or name.startswith("fc") or name.startswith("classifier"):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return [
        {"params": backbone_params, "lr": base_lr * backbone_mult, "weight_decay": weight_decay},
        {"params": head_params,     "lr": base_lr,                 "weight_decay": weight_decay},
    ]


def build_optimizer(model, arch_cfg):
    base_lr = arch_cfg["lr"]
    if arch_cfg.get("layer_decay"):
        groups, n_layers = build_vit_param_groups(model, base_lr, arch_cfg["layer_decay"])
        print(f"  ViT LLRD: {n_layers} Layers, decay={arch_cfg['layer_decay']}, {len(groups)} groups")
    else:
        groups = build_convnext_param_groups(model, base_lr, arch_cfg["lr_backbone_mult"])
        n_head = sum(len(g["params"]) for g in groups if g["lr"] == base_lr)
        n_bb = sum(len(g["params"]) for g in groups if g["lr"] != base_lr)
        print(f"  Optimizer: {n_bb} backbone (LR={base_lr*arch_cfg['lr_backbone_mult']:.1e}), "
              f"{n_head} head (LR={base_lr:.1e})")
    return optim.AdamW(groups, lr=base_lr)

## Loss: CE + Class Weights (V3-Stil)

In [8]:
def build_criterion(class_weights, label_smooth=0.05):
    """V3-Stil CE mit class weights + label smoothing."""
    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smooth)


def soft_ce(logits, targets_soft):
    """CE fuer Soft-Targets aus MixUp/CutMix (one-hot bereits gemischt)."""
    log_probs = F.log_softmax(logits, dim=-1)
    return -(targets_soft * log_probs).sum(dim=-1).mean()

## Training Helpers (mit EMA, MixUp/CutMix, Grad-Accum)

In [9]:
def train_one_epoch(model, loader, optimizer, scaler, criterion,
                    mixup_fn=None, ema=None, grad_accum=1, class_weights=None):
    model.train()
    optimizer.zero_grad()
    loss_sum, n = 0.0, 0
    preds, labels_all = [], []

    for step, (imgs, labels) in enumerate(tqdm(loader, desc="  Train", leave=False)):
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        # MixUp/CutMix -> soft targets
        if mixup_fn is not None:
            imgs_m, targets_m = mixup_fn(imgs, labels)
            with autocast():
                logits = model(imgs_m)
                # Klassen-Gewichte in soft CE einweben (manuell, timm Mixup gibt one-hot zurueck)
                if class_weights is not None:
                    log_probs = F.log_softmax(logits, dim=-1)
                    w = class_weights.unsqueeze(0)
                    loss = -(targets_m * log_probs * w).sum(dim=-1).mean()
                else:
                    loss = soft_ce(logits, targets_m)
                loss = loss / grad_accum
        else:
            with autocast():
                logits = model(imgs)
                loss = criterion(logits, labels) / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if ema is not None:
                ema.update(model)

        loss_sum += loss.item() * imgs.size(0) * grad_accum
        n += imgs.size(0)
        preds.extend(logits.argmax(1).detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return loss_sum / max(n, 1), f1_score(labels_all, preds, average="macro", zero_division=0)


@torch.no_grad()
def validate_epoch(model_or_ema, loader, criterion, ema_on_cpu=False):
    """Validiert mit gegebenem Model (kann auch EMA-State sein)."""
    model_or_ema.eval()
    loss_sum, n = 0.0, 0
    preds, labels_all = [], []
    target_device = torch.device("cpu") if ema_on_cpu else DEVICE
    for imgs, labels in tqdm(loader, desc="  Val  ", leave=False):
        imgs  = imgs.to(target_device, non_blocking=True)
        labels = labels.to(target_device, non_blocking=True)
        with autocast(enabled=not ema_on_cpu):
            logits = model_or_ema(imgs)
            loss = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        n += imgs.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    return (loss_sum / max(n, 1),
            f1_score(labels_all, preds, average="macro", zero_division=0),
            preds, labels_all)

## Strict CLO Folds (nur Test-Kameras)

Nur 3 Folds auf Test-Kameras - Validation spiegelt Kaggle-T2 exakt wider.

In [10]:
df_train = df.copy()

available_cams = set(df_train["camera"].unique())
test_cams_in_data = sorted([c for c in TEST_CAMERAS if c in available_cams])

if len(test_cams_in_data) == 0:
    print(f"T1-Modus: keine Test-Kameras -> CLO auf alle Kameras")
    cams_for_clo = sorted(available_cams)
else:
    cams_for_clo = list(test_cams_in_data)
    print(f"STRICT CLO auf Test-Kameras: {cams_for_clo}")

splits = []
fold_cameras = []
for cam in cams_for_clo:
    val_mask = df_train["camera"] == cam
    val_idx = df_train.index[val_mask].values
    train_idx = df_train.index[~val_mask].values
    if len(val_idx) > 0:
        splits.append((train_idx, val_idx))
        fold_cameras.append(cam)

n_folds = len(splits)
print(f"\n{n_folds} Folds:")
for i, cam in enumerate(fold_cameras):
    cnt = (df_train["camera"] == cam).sum()
    print(f"  Fold {i+1}: Val = {cam} ({cnt} Instanzen) | Train = {len(df_train) - cnt}")

STRICT CLO auf Test-Kameras: ['pen1_tur_cam1', 'pen2_orb_cam2', 'pen2_tur_cam2']

3 Folds:
  Fold 1: Val = pen1_tur_cam1 (200 Instanzen) | Train = 23250
  Fold 2: Val = pen2_orb_cam2 (120 Instanzen) | Train = 23330
  Fold 3: Val = pen2_tur_cam2 (196 Instanzen) | Train = 23254


## Training Loop (V5)

In [12]:
all_results = {}

for arch_idx, arch in enumerate(ARCH_LIST):
    print(f"\n{'#'*60}")
    print(f"  ARCHITEKTUR {arch_idx+1}/{len(ARCH_LIST)}: {arch['prefix']} ({arch['name']})")
    print(f"{'#'*60}")

    IMG_SIZE_ARCH  = arch["img_size"]
    BATCH_SIZE     = arch["batch_size"]
    GRAD_ACCUM     = arch.get("grad_accum", 1)
    LR             = arch["lr"]
    EPOCHS_ARCH    = arch["epochs"]
    PATIENCE       = arch["patience"]
    PREFIX         = arch["prefix"]

    arch_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        print(f"\n{'='*60}")
        print(f"  [{PREFIX}] FOLD {fold_idx + 1} / {n_folds}  (Val={fold_cameras[fold_idx]})")
        print(f"{'='*60}")

        fold_train = df_train.iloc[train_idx].reset_index(drop=True)
        fold_val   = df_train.iloc[val_idx].reset_index(drop=True)

        print(f"  Train: {len(fold_train)} | Val: {len(fold_val)} | effBatch={BATCH_SIZE*GRAD_ACCUM}")

        ckpt_path = os.path.join(OUTPUT_DIR, f"best_{PREFIX}_fold_{fold_idx+1}.pth")
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location="cpu")
            print(f"  Checkpoint existiert (val_f1={ckpt.get('val_f1',0):.4f}), skip")
            arch_results.append(ckpt.get("val_f1", 0))
            continue

        train_ds = PigPostureDataset(fold_train, transform=get_train_transform(IMG_SIZE_ARCH),
                                     pad_ratio=PAD_RATIO, is_train=True, hflip_prob=0.5)
        val_ds   = PigPostureDataset(fold_val, transform=get_val_transform(IMG_SIZE_ARCH),
                                     pad_ratio=PAD_RATIO, is_train=False)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                                  persistent_workers=(NUM_WORKERS > 0))
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True,
                                  persistent_workers=(NUM_WORKERS > 0))

        # --- Modell ---
        try:
            model = timm.create_model(arch["name"], pretrained=True,
                                      num_classes=NUM_CLASSES, img_size=IMG_SIZE_ARCH)
        except TypeError:
            model = timm.create_model(arch["name"], pretrained=True, num_classes=NUM_CLASSES)

        if PRETRAINED_CKPT and os.path.exists(PRETRAINED_CKPT):
            ck = torch.load(PRETRAINED_CKPT, map_location="cpu")
            model.load_state_dict(ck["model"], strict=False)

        model = model.to(DEVICE)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)

        params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"  Modell: {arch['name']} ({params:.1f}M)")

        # --- EMA (VOR DataParallel-Referenz, nach Model-Build) ---
        ema = ModelEMA(model, decay=EMA_DECAY, cpu=False) if USE_EMA else None
        if ema is not None:
            print(f"  EMA aktiv (decay={EMA_DECAY})")

        # --- Class weights ---
        counts = Counter(fold_train["class_id"].tolist())
        class_weights = torch.tensor(
            [len(fold_train) / (NUM_CLASSES * max(counts.get(c, 1), 1))
             for c in range(NUM_CLASSES)], dtype=torch.float32
        ).to(DEVICE)
        criterion = build_criterion(class_weights, label_smooth=LABEL_SMOOTH)
        print(f"  Loss: CE | Klassen-Gewichte: {[f'{w:.2f}' for w in class_weights.cpu().tolist()]}")

        # --- MixUp/CutMix ---
        mixup_fn = Mixup(
            mixup_alpha=MIXUP_ALPHA, cutmix_alpha=CUTMIX_ALPHA,
            prob=MIX_PROB, switch_prob=SWITCH_PROB,
            label_smoothing=LABEL_SMOOTH, num_classes=NUM_CLASSES,
        )
        print(f"  MixUp a={MIXUP_ALPHA} / CutMix a={CUTMIX_ALPHA} / prob={MIX_PROB}")

        # --- Optimizer + Scheduler ---
        optimizer = build_optimizer(model, arch)
        warmup = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
        cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS_ARCH - WARMUP_EPOCHS, eta_min=1e-7)
        scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
        scaler = GradScaler()

        best_val_f1 = 0.0
        patience_counter = 0
        best_source = "model"

        for epoch in range(1, EPOCHS_ARCH + 1):
            train_loss, train_f1 = train_one_epoch(
                model, train_loader, optimizer, scaler, criterion,
                mixup_fn=mixup_fn, ema=ema, grad_accum=GRAD_ACCUM, class_weights=class_weights
            )
            # Validate BOTH model and EMA, take the better
            val_loss, val_f1, val_preds, val_labels = validate_epoch(model, val_loader, criterion)
            source = "model"
            if ema is not None:
                e_loss, e_f1, e_preds, e_labels = validate_epoch(ema.ema, val_loader, criterion)
                if e_f1 > val_f1:
                    val_loss, val_f1, val_preds, val_labels = e_loss, e_f1, e_preds, e_labels
                    source = "ema"
            scheduler.step()

            improved = val_f1 > best_val_f1
            mark = "*" if improved else " "
            phase = "warmup" if epoch <= WARMUP_EPOCHS else "cosine"
            print(f"  {mark} Epoch {epoch:02d}/{EPOCHS_ARCH} [{phase}] | "
                  f"Train L={train_loss:.4f} F1={train_f1:.4f} | "
                  f"Val[{source}] L={val_loss:.4f} F1={val_f1:.4f}"
                  f"{' <- BEST' if improved else ''}")

            if improved:
                best_val_f1 = val_f1
                best_source = source
                patience_counter = 0
                if source == "ema" and ema is not None:
                    state = ema.state_dict()
                else:
                    state = (model.module.state_dict() if isinstance(model, nn.DataParallel)
                             else model.state_dict())
                torch.save({
                    "epoch": epoch, "model": state,
                    "val_f1": val_f1, "model_name": arch["name"], "arch_prefix": PREFIX,
                    "tag": TAG, "img_size": IMG_SIZE_ARCH, "pad_ratio": PAD_RATIO,
                    "val_camera": fold_cameras[fold_idx], "source": source,
                }, ckpt_path)
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"  Early Stopping nach {PATIENCE} Epochen")
                    break

        print(f"\n  Klassifikation Fold {fold_idx+1} (source={best_source}):")
        for c in range(NUM_CLASSES):
            mask = np.array(val_labels) == c
            if mask.sum() > 0:
                correct = (np.array(val_preds)[mask] == c).sum()
                print(f"    {CLASS_NAMES[c]:<22} {correct}/{mask.sum()} ({100*correct/max(mask.sum(),1):.0f}%)")

        arch_results.append(best_val_f1)
        print(f"  Best Val F1: {best_val_f1:.4f}  ({best_source})")

        # --- Aggressive Cleanup zwischen Folds (verhindert OOM bei Fold 2+) ---
        try:
            del train_loader, val_loader, train_ds, val_ds
        except NameError:
            pass
        del model, optimizer, scheduler, scaler
        if ema is not None:
            del ema
        import gc
        gc.collect()
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() if torch.cuda.is_available() else None

    all_results[PREFIX] = arch_results
    mean_f1 = float(np.mean(arch_results))
    print(f"\n  [{PREFIX}] Durchschnitt ueber {len(arch_results)} Folds: {mean_f1:.4f}")

print(f"\n{'#'*60}")
print(f"  GESAMT-ERGEBNIS V6")
print(f"{'#'*60}")
for prefix, results in all_results.items():
    print(f"  {prefix:<12} | Mean F1: {np.mean(results):.4f} (+/- {np.std(results):.4f}) | Folds: {results}")


############################################################
  ARCHITEKTUR 1/2: eva02l (eva02_large_patch14_448.mim_m38m_ft_in22k_in1k)
############################################################

  [eva02l] FOLD 1 / 3  (Val=pen1_tur_cam1)
  Train: 23250 | Val: 200 | effBatch=32
  Checkpoint existiert (val_f1=0.8722), skip

  [eva02l] FOLD 2 / 3  (Val=pen2_orb_cam2)
  Train: 23330 | Val: 120 | effBatch=32
  Checkpoint existiert (val_f1=0.9209), skip

  [eva02l] FOLD 3 / 3  (Val=pen2_tur_cam2)
  Train: 23254 | Val: 196 | effBatch=32
  Modell: eva02_large_patch14_448.mim_m38m_ft_in22k_in1k (304.1M)
  EMA aktiv (decay=0.9995)
  Loss: CE | Klassen-Gewichte: ['1.52', '1.36', '6.80', '0.47', '0.74']
  MixUp a=0.2 / CutMix a=1.0 / prob=0.5
  ViT LLRD: 24 Layers, decay=0.75, 50 groups


  Train:   0%|          | 0/1453 [00:00<?, ?it/s]

  Val  :   0%|          | 0/13 [00:00<?, ?it/s]

  Val  :   0%|          | 0/13 [00:00<?, ?it/s]

  * Epoch 01/30 [warmup] | Train L=1.6402 F1=0.2485 | Val[model] L=1.4882 F1=0.2647 <- BEST


...

  Val  :   0%|          | 0/13 [00:00<?, ?it/s]

  Val  :   0%|          | 0/13 [00:00<?, ?it/s]

    Epoch 12/30 [cosine] | Train L=0.6280 F1=0.8182 | Val[ema] L=0.8340 F1=0.8388


  Train:   0%|          | 0/1453 [00:00<?, ?it/s]

KeyboardInterrupt: 